# NBN benchmark — Learning curves (parameter learning vs n_train)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giovannibriglia/NeuralBayesianNetworks/blob/master/notebooks/learning_curves.ipynb)

Parameter-learning benchmark that sweeps the training-set size `n_train_sweep` [64 … 65536] on synthetic networks (n=30, three families, five seeds). Emits held-out log-likelihood, parameter recovery (TV / KL vs the true CPTs) for discrete families and predictive calibration (PIT-KS, SD-ratio) for continuous families.

| | |
|---|---|
| Config | `nbn/bench/configs/synthetic/learning_curves/learning_curves.yaml` |
| CLI subcommand | `nbn-bench param-learning` |
| Results dir | `results/benchmark_synthetic_learning_curves_<timestamp>/` |
| Expected runtime | Several hours on a T4 (ten n_train values × 14 baselines × 15 problems). |

**GPU:** this notebook is tagged for a GPU runtime, so Colab should offer one automatically when you open it.
If cell 1 reports no GPU, go to **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU** and re-run from the top.

**Session limits:** Colab recycles idle / long sessions. The runner appends one JSONL line per finished cell, so a
disconnect keeps every cell completed so far, but the parquet + figures are produced only at the end. Mount Google
Drive in section 4 to keep a copy of the results outside the ephemeral VM. For unattended multi-day runs prefer a
server (`docs/SERVER_RUN.md`).

Sections: 1 runtime check → 2 install → 3 device check → 4 (optional) Drive → 5 launch → 6 inspect → 7 plot → 8 save.


## 1. Runtime check

Confirms a CUDA GPU is attached *before* spending minutes on installation.

In [ ]:
import shutil
import subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Google Colab:", IN_COLAB)

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    raise SystemExit(
        "No NVIDIA GPU visible. In Colab: Runtime > Change runtime type > "
        "Hardware accelerator > GPU, then re-run this cell."
    )

## 2. Install

Clones the repository (full history, so the package version recorded in the results is exact) and installs it with
**all four extras** — `bench`, `neural`, `gp`, `mcmc`. Omitting any of them makes the corresponding baselines emit
`not_supported` rows instead of failing loudly, so do not trim this line. Colab ships a CUDA-enabled torch that
satisfies the `torch>=2.2` requirement; it is left untouched.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Giovannibriglia/NeuralBayesianNetworks.git"
BRANCH = "master"           # change to test a feature branch
REPO_DIR = Path("/content/NeuralBayesianNetworks") if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (REPO_DIR / "pyproject.toml").exists():
        !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        print("Repository already present; pulling latest", BRANCH)
        !git -C {REPO_DIR} pull --ff-only
else:
    # Running locally: walk up to the repo root so relative config paths resolve.
    for parent in [REPO_DIR, *REPO_DIR.parents]:
        if (parent / "pyproject.toml").exists():
            REPO_DIR = parent
            break
    else:
        raise SystemExit(f"no pyproject.toml above {REPO_DIR}; run this notebook from inside the repo")

os.chdir(REPO_DIR)
print("Working directory:", Path.cwd())
!git log -1 --format='%h %ad %s' --date=short

In [ ]:
%pip install -q -e ".[bench,neural,gp,mcmc]"
print("install done")

## 3. Device check

Verifies that torch sees the GPU and that every optional dependency imports. Stop here and fix the install if
anything fails — a multi-hour run with a silently missing extra is the expensive way to find out.

In [ ]:
import importlib

import psutil
import torch

print("torch", torch.__version__, "| CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("torch cannot see a CUDA device — switch the runtime to GPU (see section 1).")
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB")

# Quick kernel smoke test on the device
x = torch.randn(1024, 1024, device="cuda")
print("matmul on cuda OK:", (x @ x).sum().item() != 0)

print(f"Host RAM: {psutil.virtual_memory().total / 1024**3:.1f} GiB total, "
      f"{psutil.virtual_memory().available / 1024**3:.1f} GiB available")

missing = []
for mod in ["nbn", "nbn.bench", "zuko", "gpytorch", "pyro", "pgmpy", "pomegranate",
            "pandas", "pyarrow", "matplotlib", "seaborn"]:
    try:
        m = importlib.import_module(mod)
        print(f"  ok  {mod:12s} {getattr(m, '__version__', '')}")
    except Exception as exc:  # noqa: BLE001
        missing.append(mod)
        print(f"  FAIL {mod:12s} {exc}")
if missing:
    raise SystemExit(f"Missing imports: {missing} — re-run the install cell.")
!python -m nbn.bench.cli --help | head -n 5

## 4. (Optional) Google Drive

Set `USE_DRIVE = True` to mount Drive. Section 8 then copies the run directory (parquet, JSONL, `run.log`) and the
figures into `MyDrive/nbn_benchmarks/<experiment>/`. Leave `False` to skip; you can still download a zip at the end.

In [ ]:
USE_DRIVE = False
DRIVE_DIR = None

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/nbn_benchmarks/learning_curves")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Results will be copied to", DRIVE_DIR)
else:
    print("Drive not mounted; results stay on the runtime VM until you download them (section 8).")

## 5. Launch

Runs `nbn-bench param-learning` on the config through `python -m nbn.bench.cli` (identical to the console script,
but independent of `PATH`). `--device auto` resolves to CUDA for every baseline without an explicit `device:` in
the YAML; baselines pinned to `cpu` (pyro) or `cuda` (kde / knn / flexcode) keep their pin.

The console shows the tqdm progress bar plus warnings (the bar needs a terminal, which Colab's `!` provides;
do not pipe the command through `tee` or the bar disappears). The full INFO stream, including per-cell
subprocess stderr, lands in `run.log` inside the results dir. Each cell runs in its own subprocess with
a memory cap, so an OOM in one cell becomes an `oom` row rather than killing the run.

Expected runtime: Several hours on a T4 (ten n_train values × 14 baselines × 15 problems).

In [ ]:
CONFIG = "nbn/bench/configs/synthetic/learning_curves/learning_curves.yaml"
SUBCOMMAND = "param-learning"
DEVICE = "auto"        # 'auto' | 'cuda' | 'cpu'

assert Path(CONFIG).exists(), f"config not found: {CONFIG} (is the working directory the repo root?)"
!python -u -m nbn.bench.cli {SUBCOMMAND} --config {CONFIG} --device {DEVICE}

## 6. Inspect the results

Locates the newest results directory for this config, tails `run.log`, and summarises the parquet (row count and
per-status counts per baseline). `ok` rows carry metrics; `timeout` / `oom` / `error` / `not_supported` /
`not_applicable` are sentinel rows that the plotting step turns into the success-rate figure.

In [ ]:
import pandas as pd

RESULTS_ROOT = Path("results")
candidates = sorted(RESULTS_ROOT.glob("benchmark_synthetic_learning_curves_*"))
if not candidates:
    raise SystemExit("No results directory found — did the launch cell finish?")
RUN_DIR = candidates[-1]
print("Run directory:", RUN_DIR)
print("Contents:", sorted(p.name for p in RUN_DIR.iterdir()))

print("\n--- tail of run.log ---")
!tail -n 20 {RUN_DIR}/run.log

parquets = sorted(RUN_DIR.glob("*_metrics.parquet"))
if not parquets:
    raise SystemExit(
        "No *_metrics.parquet in the run dir. The run may have been interrupted; the per-cell JSONL is still there. "
        "Convert it with: from nbn.bench.core.output import jsonl_to_parquet; "
        f"jsonl_to_parquet(Path('{RUN_DIR}/metrics.jsonl'), Path('{RUN_DIR}/learning_curves_metrics.parquet'))"
    )
PARQUET = parquets[0]
df = pd.read_parquet(PARQUET)
print(f"\n{PARQUET.name}: {len(df)} rows, {df.shape[1]} columns")
if "status" in df.columns:
    print("\nStatus counts:")
    print(df["status"].value_counts().to_string())
    if "baseline" in df.columns:
        print("\nStatus per baseline:")
        print(pd.crosstab(df["baseline"], df["status"]).to_string())
df.head()

## 7. Plot

`nbn-bench plot` reads the parquet and writes the paper figures (PDF) and LaTeX tables per family and coverage
subset into `--output-dir` (layout in `docs/v0.13-paper-figures.md`). The second cell renders the PDFs inline so
you can eyeball them without downloading anything.

In [ ]:
FIG_DIR = RUN_DIR / "figures"
AGGREGATION = "iqm_iqr"     # 'iqm_iqr' (paper default) | 'mean_std'

!python -m nbn.bench.cli plot {PARQUET} --output-dir {FIG_DIR} --aggregation {AGGREGATION}

pdfs = sorted(FIG_DIR.rglob("*.pdf"))
texs = sorted(FIG_DIR.rglob("*.tex"))
print(f"\n{len(pdfs)} figures, {len(texs)} LaTeX tables under {FIG_DIR}")
overview = FIG_DIR.rglob("_subsets_overview.txt")
for ov in overview:
    print(f"\n--- {ov.relative_to(FIG_DIR)} ---")
    print(ov.read_text())

In [ ]:
# Render the PDF figures inline (PyMuPDF rasterises them; no poppler needed).
%pip install -q pymupdf
import pymupdf
from IPython.display import Image, Markdown, display

MAX_FIGURES = 40          # raise if you want to see every subset
SHOW_ONLY = "all"         # 'all' shows the mixed-coverage subset; None shows every subset

shown = 0
for pdf in pdfs:
    if SHOW_ONLY and SHOW_ONLY not in pdf.parts:
        continue
    if shown >= MAX_FIGURES:
        print(f"... {len(pdfs) - shown} more figures not shown (raise MAX_FIGURES)")
        break
    page = pymupdf.open(pdf)[0]
    png = page.get_pixmap(dpi=110).tobytes("png")
    display(Markdown(f"**{pdf.relative_to(FIG_DIR)}**"))
    display(Image(data=png))
    shown += 1
print(f"Displayed {shown} figures.")

## 8. Save the artifacts

Zips the run directory (parquet, JSONL, `run.log`, figures, tables) and either copies it to Google Drive (if
mounted in section 4) or triggers a browser download. The zip is the complete, reproducible record of this run.

In [ ]:
import shutil

ARCHIVE = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print("Archive:", ARCHIVE, f"({Path(ARCHIVE).stat().st_size / 1024**2:.1f} MiB)")

if DRIVE_DIR is not None:
    dest = DRIVE_DIR / Path(ARCHIVE).name
    shutil.copy2(ARCHIVE, dest)
    print("Copied to Drive:", dest)
elif IN_COLAB:
    from google.colab import files
    files.download(ARCHIVE)
else:
    print("Not in Colab; archive left at", ARCHIVE)